In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: sales1
position:
  x: 0
  y: 0
description:
  text: Read data from a CSV file with specified options.
  hash: f6410f4f
previewCodeHash: a9425c903afaceb6
previewMode: "1000"
config:
  file_source:
    path: /Volumes/dev/demo/raw-1000-richest/sales/sales1.csv
    format: '"csv"'
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/dev/demo/raw-1000-richest/sales/sales1.csv",
        "format": "\"csv\""
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_0.data"])

In [0]:
"""
id: filter_1
template: filter
templateVersion: 2.0.0
name: remove_nulls
position:
  x: 271.48148148148147
  y: 264.76851851851853
description:
  text: Keep rows where order, customer, transaction, and product IDs are all present; separate out rows missing any of these IDs.
  hash: 15dc7833
previewCodeHash: 457494c251d7a07d
previewMode: "1000"
config:
  condition: order_id IS NOT NULL AND customer_id IS NOT NULL AND transaction_id IS NOT NULL AND product_id IS NOT NULL
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "order_id IS NOT NULL AND customer_id IS NOT NULL AND transaction_id IS NOT NULL AND product_id IS NOT NULL"
}
inputs = {
    "data": ctx["source_0.data"]
}
out = run(config, inputs, spark)
ctx["filter_1.filtered_data"] = out["filtered_data"]
ctx["filter_1.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["filter_1.filtered_data"])
    display(ctx["filter_1.excluded_data"])

In [0]:
"""
id: unique_1a0cdb23
template: unique
templateVersion: 1.0.0
name: Drop duplicates
position:
  x: 406.8518518518518
  y: -1.2962962962963047
previewCodeHash: f7aaaede18010905
previewMode: "1000"
config:
  unique_by_all_columns: true
  columns: []
  sort_expressions: []
input:
  - node: filter_1
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F
from pyspark.sql import Window

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError("Unique operator requires input 'data'")
    unique_by_all_columns = config.get("unique_by_all_columns", True)
    columns = config.get("columns", [])
    if not isinstance(columns, list):
        columns = []
    else:
        columns = [c for c in columns if isinstance(c, str)]
    sort_expressions = config.get("sort_expressions", [])

    if unique_by_all_columns:
        columns = df.columns
    elif not columns:
        return {"unique_data": df}

    order_idx = "__lb_orig_order__"
    while order_idx in df.columns:
        order_idx = order_idx + "_"
    df = df.withColumn(order_idx, F.monotonically_increasing_id())

    if unique_by_all_columns or not sort_expressions:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    order_cols = []
    for sort_def in sort_expressions:
        if not isinstance(sort_def, dict):
            continue
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        if not raw_expr:
            continue
        direction = sort_def.get("sortBy", "ASC")
        col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        if direction == "DESC":
            col = col.desc_nulls_last()
        elif direction == "ASC":
            col = col.asc_nulls_last()
        order_cols.append(col)
    
    if not order_cols:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    rn = "__lb_row_number__"
    while rn in df.columns:
        rn = rn + "_"

    window = Window.partitionBy(*[F.col(c) for c in columns]).orderBy(*order_cols, F.col(order_idx))
    tagged = df.withColumn(rn, F.row_number().over(window))
    unique = tagged.filter(F.col(rn) == 1).orderBy(order_idx).drop(rn, order_idx)
    return {"unique_data": unique}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "unique_by_all_columns": True,
    "columns": [],
    "sort_expressions": []
}
inputs = {
    "data": ctx["filter_1.filtered_data"]
}
out = run(config, inputs, spark)
ctx["unique_1a0cdb23.unique_data"] = out["unique_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["unique_1a0cdb23.unique_data"])

In [0]:
"""
id: filter_3
template: filter
templateVersion: 2.0.0
name: valid_amount
position:
  x: 700.3703703703703
  y: 199.62962962962965
previewMode: "1000"
config:
  condition: total_amount > 0
input:
  - node: unique_1a0cdb23
    input_port: data
    output_port: unique_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "total_amount > 0"
}
inputs = {
    "data": ctx["unique_1a0cdb23.unique_data"]
}
out = run(config, inputs, spark)
ctx["filter_3.filtered_data"] = out["filtered_data"]
ctx["filter_3.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["filter_3.filtered_data"])
    display(ctx["filter_3.excluded_data"])

In [0]:
"""
id: output_4
template: output
templateVersion: 3.0.0
name: clean_sales
position:
  x: 865.8796296296296
  y: -10.37037037037037
description:
  text: Overwrite table visual_prep_silver in cyntexa_dev.sales with new data.
  hash: 2fdab103
previewMode: "1000"
config:
  output_type: table
  catalog: cyntexa_dev
  schema: sales
  table_name: visual_prep_silver
  write_mode: overwrite
input:
  - node: filter_3
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "cyntexa_dev",
    "schema": "sales",
    "table_name": "visual_prep_silver",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["filter_3.filtered_data"]
}
out = run(config, inputs, spark)